In [1]:
import os, sys

import torch
import numpy as np
import networkx as nx

import matplotlib.pyplot as plt 

from init_graph import *


In [2]:
# Import seaborn
import seaborn as sns

# Apply the default theme
sns.set_theme()

plt.rcParams["figure.dpi"]=100
plt.rcParams['savefig.dpi']=300

# Set a modern style for the plot
sns.set_theme(style="whitegrid")
sns.set_style({"axes.facecolor": ".98"})
colors = sns.color_palette("muted", 7)
labels = ["$FairRARI$", "PR+proj", "Inbetween"]

In [ ]:
logs_source_path_FairRARI = "logs/FairRARI/"
logs_source_path_post_processing = "logs/post_processing/"

save_source_path = "figures/"
save_ = 1

In [4]:
dataset_name = 'polbooks'

In [ ]:
source_path = "datasets/"

G, protected_nodes, blue_nodes, red_nodes = init_graph(dataset_name, source_path)
unprotected_nodes = blue_nodes


n = G.number_of_nodes()
m = G.number_of_edges()
print('Number of Nodes:', n)
print('Number of Edges:', m)

# Create Protected and Un-Protected Set Vectors
S_p = torch.zeros(n).int()
S_p[protected_nodes] = 1
S_up = torch.ones(n).int()
S_up[protected_nodes] = 0
n_p = len(protected_nodes)
print('Number of Protected Nodes:', n_p)
print('Number of UnProtected Nodes:', len(unprotected_nodes))

print('Number of Protected Nodes over Number of Nodes: {:.6f}'.format(n_p/n))

In [ ]:
gamma = 0.15
max_iters_eucl_iters = 1000
max_iters_eucl_iters_1proj = 1000

In [ ]:
phi = 0.9

In [ ]:
dir_path_FairRARI = logs_source_path_FairRARI+dataset_name+'/'

load_path_FairRARI = dir_path_FairRARI+dataset_name+'_phi'+'{:.2f}'.format(phi)+'_gamma'+'{:.2f}'.format(gamma)
load_path_FairRARI = load_path_FairRARI + '_iters'+str(max_iters_eucl_iters)+'_log.npy'

variables_dict_FairRARI = np.load(load_path_FairRARI, allow_pickle=True)

FairRARI_scores = variables_dict_FairRARI.item().get('x_opt')
opr_scores = variables_dict_FairRARI.item().get('opr_scores')

In [ ]:
dir_path_post_processing = logs_source_path_post_processing+dataset_name+'/'

load_path_post_processing = dir_path_post_processing+dataset_name+'_phi'+'{:.2f}'.format(phi)+'_gamma'+'{:.2f}'.format(gamma)
load_path_post_processing = load_path_post_processing + '_iters'+str(max_iters_eucl_iters_1proj)+'_log.npy'

variables_dict_post_processing = np.load(load_path_post_processing, allow_pickle=True)

post_processing_scores = variables_dict_post_processing.item().get('x_opt')

# Network Plots

In [ ]:
pos = nx.spring_layout(G, seed=0)  # positions for all nodes

In [ ]:
zero_pr_nodes = np.where(FairRARI_scores == 0)[0].tolist()

# Define node colors based on the protected/unprotected sets
node_colors = ['C3' if node in protected_nodes else 'C0' for node in G.nodes()]
node_colors = ['k' if FairRARI_scores[node]==0 else node_colors[node] for node in G.nodes()]
unprotected_nodes = blue_nodes

# Draw the network
# You can adjust the layout, but here I am using a spring layout
edge_colors = []
for u, v in G.edges():
    if u in protected_nodes and v in protected_nodes:
        edge_colors.append('C3')  # Same class (protected)
    elif u in unprotected_nodes and v in unprotected_nodes:
        edge_colors.append('C0')  # Same class (unprotected)
    else:
        edge_colors.append('C4')  # Different classes (combination of blue and red)

# Define node sizes based on PageRank scores (scaled for better visibility)
node_sizes = [FairRARI_scores[node] * 1e4 for node in G.nodes()]  # Scaling for better visibility
# node_sizes = [0.001e4 if pagerank_scores_fairrari[node]==0 else node_sizes[node] for node in G.nodes()]
# Define dynamic font sizes based on node sizes
max_node_size = max(node_sizes)
font_sizes = [size / max_node_size * 15 + 5 for size in node_sizes]  # Scale and offset for visibility

# Create a plot
# plt.figure(figsize=(10, 8))
plt.figure(figsize=(6, 4.8))

# Draw the network
nx.draw(
    G,
    pos,
    with_labels=False,
    # node_shape=node_shapes,
    node_color=node_colors,  # Node color based on class
    node_size=node_sizes,    # Node size based on PageRank
    edge_color=edge_colors,  # Edge colors based on connected nodes' classes
    alpha=0.85               # Transparency for better visibility
)

nx.draw_networkx_nodes(G, pos, nodelist=zero_pr_nodes, node_color='none', edgecolors='k', node_shape='o', node_size=0.003e4)


plt.title("FairRARI")

save_folder = save_source_path +'network/'+dataset_name+'/'
save_path = save_folder+dataset_name+'_network_'+str(phi)
save_path = save_path+'_FairRARI.pdf' 
if save_==1:
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    plt.savefig(save_path, format='pdf', bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
zero_pr_nodes = np.where(post_processing_scores == 0)[0].tolist()

# Define node colors based on the protected/unprotected sets
node_colors = ['C3' if node in protected_nodes else 'C0' for node in G.nodes()]
node_colors = ['k' if post_processing_scores[node]==0 else node_colors[node] for node in G.nodes()]
unprotected_nodes = blue_nodes

# Draw the network
# You can adjust the layout, but here I am using a spring layout
edge_colors = []
for u, v in G.edges():
    if u in protected_nodes and v in protected_nodes:
        edge_colors.append('C3')  # Same class (protected)
    elif u in unprotected_nodes and v in unprotected_nodes:
        edge_colors.append('C0')  # Same class (unprotected)
    else:
        edge_colors.append('C4')  # Different classes (combination of blue and red)

# Define node sizes based on PageRank scores (scaled for better visibility)
node_sizes = [post_processing_scores[node] * 1e4 for node in G.nodes()]  # Scaling for better visibility
# Define dynamic font sizes based on node sizes
max_node_size = max(node_sizes)
font_sizes = [size / max_node_size * 15 + 5 for size in node_sizes]  # Scale and offset for visibility

# Create a plot
# plt.figure(figsize=(10, 8))
plt.figure(figsize=(6, 4.8))

# Draw the network
nx.draw(
    G,
    pos,
    with_labels=False,
    # node_shape=node_shapes,
    node_color=node_colors,  # Node color based on class
    node_size=node_sizes,    # Node size based on PageRank
    edge_color=edge_colors,  # Edge colors based on connected nodes' classes
    alpha=0.85               # Transparency for better visibility
)

nx.draw_networkx_nodes(G, pos, nodelist=zero_pr_nodes, node_color='none', edgecolors='k', node_shape='o', node_size=0.003e4)


plt.title("Post-Processing")

save_folder = save_source_path +'network/'+dataset_name+'/'
save_path = save_folder+dataset_name+'_network_'+str(phi)
save_path = save_path+'_post_processing.pdf' 
if save_==1:
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    plt.savefig(save_path, format='pdf', bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
print("Number of Nodes with 0 Score:")
print("FairRARI:        ", sum(FairRARI_scores==0).item())
print("Post-Processing: ", sum(post_processing_scores==0).item())

In [ ]:
# Define node colors based on the protected/unprotected sets
node_colors = ['C3' if node in protected_nodes else 'C0' for node in G.nodes()]
node_colors = ['k' if opr_scores[node]==0 else node_colors[node] for node in G.nodes()]
unprotected_nodes = blue_nodes

# Draw the network
# You can adjust the layout, but here I am using a spring layout
edge_colors = []
for u, v in G.edges():
    if u in protected_nodes and v in protected_nodes:
        edge_colors.append('C3')  # Same class (protected)
    elif u in unprotected_nodes and v in unprotected_nodes:
        edge_colors.append('C0')  # Same class (unprotected)
    else:
        edge_colors.append('C4')  # Different classes (combination of blue and red)

# Define node sizes based on PageRank scores (scaled for better visibility)
node_sizes = [opr_scores[node] * 1e4 for node in G.nodes()]  # Scaling for better visibility
# Define dynamic font sizes based on node sizes
max_node_size = max(node_sizes)
font_sizes = [size / max_node_size * 15 + 5 for size in node_sizes]  # Scale and offset for visibility

# Create a plot
# plt.figure(figsize=(10, 8))
plt.figure(figsize=(6, 4.8))

# Draw the network
nx.draw(
    G,
    pos,
    with_labels=False,
    node_color=node_colors,  # Node color based on class
    node_size=node_sizes,    # Node size based on PageRank
    edge_color=edge_colors,  # Edge colors based on connected nodes' classes
    alpha=0.85               # Transparency for better visibility
)

plt.title("Original PageRank")

save_folder = save_source_path +'network/'+dataset_name+'/'
save_path = save_folder+dataset_name+'_network'
save_path = save_path+'_OPR.pdf' 
if save_==1:
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    plt.savefig(save_path, format='pdf', bbox_inches='tight')

# Show the plot
plt.show()